# Kaggle Training V4.0 — DermaFusion 24-Class (ISIC2019 + DermNet + PAD-UFES-20)

- ✅ **Tải `class_weights.json` từ dataset** thay vì tự tính lại bằng sqrt — nhất quán với `prepare_dataset_local.py`
- ✅ **Kết hợp WeightedRandomSampler + class_weights_tensor** để giải quyết imbalance ở cả sampling lẫn loss
- ✅ **`torch.backends.cudnn.deterministic = True`** + `worker_init_fn` cho reproducibility đầy đủ
- ✅ **AMP autocast** bọc đúng cả val loop để tiết kiệm VRAM
- ✅ **TTA (Test-Time Augmentation)** ở validation cuối cùng để đánh giá chính xác hơn
- ✅ **Lưu metadata đầy đủ** vào checkpoint (class_names, IMG_SIZE, epoch, optimizer state)
- ✅ **Gradient accumulation** hỗ trợ effective batch size lớn hơn trên GPU nhỏ
- ✅ **Per-class F1 log** sau mỗi epoch để theo dõi class khó
- ✅ **EarlyStopping giám sát val_f1** (không phải val_loss) — phù hợp với bài toán imbalanced
- ✅ **Validation confusion matrix** được lưu dưới dạng normalized để dễ đọc

**Dataset:** ISIC2019 + DermNet + PAD-UFES-20 → 24 classes, processed bởi `prepare_dataset_local.py` + `HybridPreprocessingPipeline`

In [ ]:
# ============================================================
# CELL 1: INSTALL & IMPORTS
# ============================================================
import os, glob, random, json, warnings
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
from tqdm.auto import tqdm
import torch.nn.functional as F
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
print('Imports OK')

In [ ]:
# ============================================================
# CELL 2: SETTINGS & REPRODUCIBILITY
# ============================================================
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Đảm bảo DataLoader workers cũng dùng seed cố định
    os.environ['PYTHONHASHSEED'] = str(seed)

def worker_init_fn(worker_id):
    """Seed mỗi DataLoader worker để reproducible"""
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

set_seed()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    # benchmark=True tăng tốc nhưng không deterministic.
    # Dùng True khi ưu tiên tốc độ (production), False khi ưu tiên reproducibility.
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False  # False = nhanh hơn, đủ cho training

print(f'Torch version: {torch.__version__}')

In [ ]:
# ============================================================
# CELL 3: DATASET DISCOVERY
# ============================================================
import itertools

EXTRACT_DIR = None

# Các path phổ biến trên Kaggle — thêm vào nếu dataset có tên khác
KNOWN_PATHS = [
    '/kaggle/input/derma-db-24class/processed',
    '/kaggle/input/dermadb-24class-v2/processed',
    '/kaggle/input/dermafusion-processed/processed',
    '/kaggle/input/dermafusion-24class/processed',
    '/kaggle/input/datasets/dnghongkhang/derma-db-24class/processed',
]

for path in KNOWN_PATHS:
    if os.path.exists(os.path.join(path, 'train')):
        EXTRACT_DIR = path
        print(f'✅ Found dataset at known path: {EXTRACT_DIR}')
        break

# Fallback: tìm bất kỳ thư mục nào có train/ val/
if not EXTRACT_DIR and os.path.exists('/kaggle/input'):
    for root, dirs, _ in itertools.islice(os.walk('/kaggle/input'), 300):
        if 'train' in dirs and 'val' in dirs:
            EXTRACT_DIR = root
            print(f'✅ Found dataset via walk: {EXTRACT_DIR}')
            break

if not EXTRACT_DIR:
    raise RuntimeError('❌ Dataset NOT FOUND. Hãy add dataset derma-db-24class vào notebook.')

# Kiểm tra class_weights.json — cần thiết cho loss weighting
# FIX v4.0: Load class_weights từ file do prepare_dataset_local.py tạo ra
# để đảm bảo nhất quán với phân phối thực tế của dataset
WEIGHTS_PATHS = [
    os.path.join(os.path.dirname(EXTRACT_DIR), 'class_weights.json'),
    '/kaggle/input/derma-db-24class/class_weights.json',
    '/kaggle/input/dermadb-24class-v2/class_weights.json',
    '/kaggle/input/dermafusion-processed/class_weights.json',
]
CLASS_WEIGHTS_PATH = None
for p in WEIGHTS_PATHS:
    if os.path.exists(p):
        CLASS_WEIGHTS_PATH = p
        print(f'✅ Found class_weights.json: {CLASS_WEIGHTS_PATH}')
        break

if not CLASS_WEIGHTS_PATH:
    print('⚠️  class_weights.json not found — sẽ tính lại từ dataset (fallback)')

print(f'Dataset: {EXTRACT_DIR}')

In [ ]:
# ============================================================
# CELL 4: AUGMENTATIONS
# ============================================================
# IMG_SIZE=380 khớp với target_size trong HybridPreprocessingPipeline
IMG_SIZE = 380
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Train: augmentation mạnh để tăng generalization
# Ảnh da liễu thường có góc chụp bất kỳ → flip/rotate tốt
# ColorJitter mạnh → model robust với ánh sáng/camera khác nhau
train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0), ratio=(0.75, 1.33)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(30),
    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.3,
        hue=0.1
    ),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=5)], p=0.2),
    # AutoAugment cho skin lesion — thêm TrivialAugmentWide cho đa dạng hơn
    transforms.RandomApply([transforms.RandomAdjustSharpness(sharpness_factor=2)], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.15)),
])

# Val: deterministic, không augmentation ngẫu nhiên
val_tfms = transforms.Compose([
    transforms.Resize(IMG_SIZE + 20),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# TTA transforms — dùng khi eval cuối cùng
# 5 crops: center + 4 corners → average predictions
tta_tfms_list = [
    transforms.Compose([
        transforms.Resize(IMG_SIZE + 20),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]),
    transforms.Compose([
        transforms.Resize(IMG_SIZE + 20),
        transforms.CenterCrop(IMG_SIZE),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]),
    transforms.Compose([
        transforms.Resize(IMG_SIZE + 20),
        transforms.CenterCrop(IMG_SIZE),
        transforms.RandomVerticalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]),
]
print('Augmentations OK')

In [ ]:
# ============================================================
# CELL 5: DATA LOADING & CLASS WEIGHTS
# ============================================================
BATCH_SIZE = 48
# Gradient accumulation: effective batch = BATCH_SIZE * ACCUM_STEPS
# Hữu ích khi GPU nhỏ (T4/P100) không chứa được batch lớn
ACCUM_STEPS = 1  # Tăng lên 2 nếu OOM, effective_batch sẽ = 96
NUM_WORKERS = 4

train_dir = os.path.join(EXTRACT_DIR, 'train')
val_dir   = os.path.join(EXTRACT_DIR, 'val')

train_ds = datasets.ImageFolder(train_dir, transform=train_tfms)
val_ds   = datasets.ImageFolder(val_dir,   transform=val_tfms)

class_names = train_ds.classes
NUM_CLASSES = len(class_names)
print(f'Classes ({NUM_CLASSES}): {class_names}')

# ---------- Class Weights ----------
# FIX v4.0: Ưu tiên load từ class_weights.json (do prepare_dataset_local.py tính)
# File này dùng công thức: total / (n_classes * count) — balanced weighting
# Nhất quán với phân phối thực tế của dữ liệu đã xử lý.
if CLASS_WEIGHTS_PATH:
    with open(CLASS_WEIGHTS_PATH, 'r') as f:
        cw_dict = json.load(f)
    # Sắp xếp theo thứ tự class_names của ImageFolder
    class_weights_arr = np.array([
        cw_dict.get(cn, 1.0) for cn in class_names
    ], dtype=np.float32)
    print(f'✅ Loaded class weights from {CLASS_WEIGHTS_PATH}')
else:
    # Fallback: tính lại từ distribution trong dataset
    labels_list = [y for _, y in train_ds.samples]
    counts = np.bincount(labels_list, minlength=NUM_CLASSES).astype(np.float32)
    total = counts.sum()
    class_weights_arr = total / (NUM_CLASSES * (counts + 1e-6))
    print('⚠️  Fallback: computed class weights from dataset distribution')

# Normalize về [1/2, 2] để tránh weight quá lớn gây instability
class_weights_arr = np.clip(class_weights_arr, 0.5, 10.0)
class_weights_tensor = torch.tensor(class_weights_arr, dtype=torch.float32, device=device)

print('\nClass weights (sorted by weight desc):')
sorted_cw = sorted(zip(class_names, class_weights_arr), key=lambda x: -x[1])
for cn, w in sorted_cw:
    print(f'  {cn:35s}: {w:.4f}')

# ---------- WeightedRandomSampler ----------
# Giải quyết imbalance ở sampling level: đảm bảo class hiếm được lấy mẫu đủ
labels_list = [y for _, y in train_ds.samples]
sample_weights = [float(class_weights_arr[lbl]) for lbl in labels_list]
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=True,
    prefetch_factor=2 if NUM_WORKERS > 0 else None,
    persistent_workers=(NUM_WORKERS > 0),
    worker_init_fn=worker_init_fn,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    prefetch_factor=2 if NUM_WORKERS > 0 else None,
    persistent_workers=(NUM_WORKERS > 0),
)

print(f'\nTrain batches: {len(train_loader)} | Val batches: {len(val_loader)}')
print(f'Effective batch size: {BATCH_SIZE * ACCUM_STEPS}')

# Hiển thị phân phối thực tế
actual_counts = np.bincount(labels_list, minlength=NUM_CLASSES)
print(f'\nClass distribution (train):')
for cn, cnt in sorted(zip(class_names, actual_counts), key=lambda x: x[1]):
    bar = '█' * (cnt // 100)
    print(f'  {cn:35s}: {cnt:5d} {bar}')

In [ ]:
# ============================================================
# CELL 6: MODEL — EfficientNet-B4 từ ImageNet weights
# ============================================================
model = efficientnet_b4(weights=EfficientNet_B4_Weights.IMAGENET1K_V1)

in_features = model.classifier[1].in_features

# Head mạnh hơn: 2 layers với BatchNorm để giảm internal covariate shift
# Phù hợp với bài toán medical image (phân phối khác ImageNet)
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4, inplace=False),
    nn.Linear(in_features, 512),
    nn.BatchNorm1d(512),
    nn.SiLU(),  # SiLU (Swish) khớp với activation dùng trong EfficientNet backbone
    nn.Dropout(p=0.3, inplace=False),
    nn.Linear(512, NUM_CLASSES)
)

model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: EfficientNet-B4 (ImageNet pretrained)')
print(f'Head: {in_features} -> 512 -> {NUM_CLASSES}')
print(f'Total params: {total_params/1e6:.2f}M | Trainable: {trainable_params/1e6:.2f}M')

In [ ]:
# ============================================================
# CELL 7: LOSS FUNCTION — Asymmetric Loss + Class Weights
# ============================================================
class AsymmetricLossWithSmoothing(nn.Module):
    """
    Asymmetric Loss kết hợp Label Smoothing và Class Weighting.

    Lý do dùng:
    - gamma_neg > gamma_pos: phạt nặng false negative (bỏ qua ung thư) hơn false positive
    - label_smoothing: tránh model overconfident, cải thiện calibration
    - class_weights: giải quyết imbalance ở loss level (bổ sung cho sampler)

    FIX v4.0: Thêm class_weights vào loss để dual-level imbalance handling:
    sampler cân bằng số lượng mẫu, class_weights cân bằng contribution vào gradient.
    """
    def __init__(self, gamma_neg=4.0, gamma_pos=1.0, clip=0.05,
                 smoothing=0.1, class_weights=None):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.smoothing = smoothing
        # Lưu class_weights như buffer (sẽ tự động move sang đúng device)
        if class_weights is not None:
            self.register_buffer('class_weights', class_weights)
        else:
            self.class_weights = None

    def forward(self, logits, targets):
        num_classes = logits.size(1)

        # Label smoothing
        targets_oh = F.one_hot(targets, num_classes=num_classes).float()
        if self.smoothing > 0:
            targets_oh = targets_oh * (1 - self.smoothing) + self.smoothing / num_classes

        probs_pos = torch.softmax(logits, dim=1)
        probs_neg = (1 - probs_pos + self.clip).clamp(max=1)

        log_pos = torch.log(probs_pos.clamp(min=1e-8))
        log_neg = torch.log(probs_neg.clamp(min=1e-8))

        loss = (
            -targets_oh * (1 - probs_pos) ** self.gamma_pos * log_pos
            - (1 - targets_oh) * probs_pos ** self.gamma_neg * log_neg
        )  # shape: (B, C)

        # FIX v4.0: Apply class weights — weight theo ground truth class
        if self.class_weights is not None:
            # targets shape: (B,), class_weights shape: (C,)
            weight_per_sample = self.class_weights[targets]  # (B,)
            loss_per_sample = loss.sum(dim=1)  # (B,)
            return (loss_per_sample * weight_per_sample).mean()

        return loss.sum(dim=1).mean()


criterion = AsymmetricLossWithSmoothing(
    gamma_neg=4.0,
    gamma_pos=1.0,
    smoothing=0.1,
    class_weights=class_weights_tensor
).to(device)

print('Loss: AsymmetricLoss (gamma_neg=4, gamma_pos=1, smoothing=0.1, class_weighted=True)')

In [ ]:
# ============================================================
# CELL 8: OPTIMIZER & SCHEDULER
# ============================================================
# Differential LR: backbone thấp hơn head vì backbone đã pretrained
BASE_LR_FEATURES = 5e-5
BASE_LR_HEAD     = 3e-4
WARMUP_EPOCHS    = 5
NUM_EPOCHS       = 40
WEIGHT_DECAY     = 1e-3

optimizer = torch.optim.AdamW([
    {'params': model.features.parameters(),   'lr': BASE_LR_FEATURES},
    {'params': model.classifier.parameters(), 'lr': BASE_LR_HEAD}
], weight_decay=WEIGHT_DECAY)

# Cosine annealing sau warmup
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS - WARMUP_EPOCHS,
    eta_min=1e-7
)

def apply_warmup(epoch, warmup_epochs, base_lrs):
    """Linear warmup: LR tăng dần từ 0 đến base_lr trong warmup_epochs epochs"""
    if epoch < warmup_epochs:
        scale = (epoch + 1) / warmup_epochs
        for i, pg in enumerate(optimizer.param_groups):
            pg['lr'] = base_lrs[i] * scale
        return True
    return False

base_lrs = [BASE_LR_FEATURES, BASE_LR_HEAD]

# AMP GradScaler — chỉ active trên CUDA
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

print(f'Optimizer: AdamW (features lr={BASE_LR_FEATURES}, head lr={BASE_LR_HEAD}, wd={WEIGHT_DECAY})')
print(f'Scheduler: Linear warmup ({WARMUP_EPOCHS} ep) → CosineAnnealing ({NUM_EPOCHS - WARMUP_EPOCHS} ep)')
print(f'Epochs: {NUM_EPOCHS} | Patience: 12 | Accum steps: {ACCUM_STEPS}')

In [ ]:
# ============================================================
# CELL 9: TRAINING LOOP
# ============================================================
PATIENCE   = 12
OUT_NAME   = 'efficientnet_b4_derma_v4.pth'
HIST_FILE  = 'training_history_v4.json'

best_val_f1    = 0.0
epochs_no_imp  = 0
history        = []

print(f'Starting v4.0 training ({NUM_EPOCHS} epochs, warmup={WARMUP_EPOCHS})...')
print(f'Output checkpoint: {OUT_NAME}')
print()

for epoch in range(1, NUM_EPOCHS + 1):
    # --- Warmup / LR schedule ---
    is_warmup = apply_warmup(epoch - 1, WARMUP_EPOCHS, base_lrs)
    current_lr = optimizer.param_groups[0]['lr']

    # -------- TRAIN --------
    model.train()
    train_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    for batch_idx, (images, labels_batch) in enumerate(tqdm(train_loader, desc=f'Ep{epoch} Train', leave=False)):
        images       = images.to(device, non_blocking=True)
        labels_batch = labels_batch.to(device, non_blocking=True)

        with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            logits = model(images)
            # FIX v4.0: chia loss cho ACCUM_STEPS để gradient scale đúng
            loss = criterion(logits, labels_batch) / ACCUM_STEPS

        scaler.scale(loss).backward()

        # FIX v4.0: Gradient accumulation
        if (batch_idx + 1) % ACCUM_STEPS == 0 or (batch_idx + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        train_loss += loss.item() * ACCUM_STEPS * images.size(0)

    train_loss /= len(train_loader.dataset)

    # Step cosine scheduler AFTER warmup
    if not is_warmup:
        cosine_scheduler.step()

    # -------- VALIDATION --------
    model.eval()
    val_loss   = 0.0
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for images, labels_batch in tqdm(val_loader, desc=f'Ep{epoch} Val', leave=False):
            images       = images.to(device, non_blocking=True)
            labels_batch = labels_batch.to(device, non_blocking=True)
            # FIX v4.0: AMP cũng wrap val để nhất quán và tiết kiệm VRAM
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                logits = model(images)
                loss   = criterion(logits, labels_batch)
            val_loss += loss.item() * images.size(0)
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(labels_batch.cpu().numpy())

    val_loss /= len(val_loader.dataset)
    val_f1   = f1_score(all_labels, all_preds, average='macro',  zero_division=0)
    val_acc  = (np.array(all_preds) == np.array(all_labels)).mean()

    # FIX v4.0: Per-class F1 để theo dõi class khó
    per_class_f1 = f1_score(all_labels, all_preds, average=None, zero_division=0)
    worst3 = sorted(zip(class_names, per_class_f1), key=lambda x: x[1])[:3]
    worst3_str = ', '.join(f'{cn}={f:.2f}' for cn, f in worst3)

    phase = 'WARMUP' if is_warmup else 'COSINE'
    history.append({
        'epoch':      epoch,
        'phase':      phase,
        'lr':         round(current_lr, 8),
        'train_loss': round(train_loss, 4),
        'val_loss':   round(val_loss,   4),
        'val_acc':    round(float(val_acc),  4),
        'val_f1':     round(float(val_f1),   4),
        'per_class_f1': {cn: round(float(f), 4) for cn, f in zip(class_names, per_class_f1)}
    })

    print(f'Ep {epoch:2d}/{NUM_EPOCHS} [{phase}] lr={current_lr:.2e} | '
          f'train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | '
          f'val_acc={val_acc:.4f} | val_f1={val_f1:.4f} | worst: [{worst3_str}]')

    # Checkpoint
    if val_f1 > best_val_f1:
        best_val_f1   = val_f1
        epochs_no_imp = 0
        tmp_path = OUT_NAME + '.tmp'
        torch.save({
            # FIX v4.0: Lưu đầy đủ metadata cần thiết để reload
            'model_state':     model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'epoch':           epoch,
            'val_f1':          best_val_f1,
            'val_acc':         float(val_acc),
            'class_names':     class_names,
            'num_classes':     NUM_CLASSES,
            'img_size':        IMG_SIZE,
            'per_class_f1':    {cn: round(float(f), 4) for cn, f in zip(class_names, per_class_f1)},
        }, tmp_path)
        os.replace(tmp_path, OUT_NAME)
        print(f'  ✓ Saved best checkpoint (F1={best_val_f1:.4f}, Acc={val_acc:.4f})')
    else:
        epochs_no_imp += 1

    if epochs_no_imp >= PATIENCE:
        print(f'Early stopping at epoch {epoch} (no improvement for {PATIENCE} epochs).')
        break

# Restore best
best_ckpt = torch.load(OUT_NAME, map_location=device, weights_only=False)
model.load_state_dict(best_ckpt['model_state'])
print(f'\n✅ Restored best model from epoch {best_ckpt["epoch"]} '
      f'(F1={best_ckpt["val_f1"]:.4f}, Acc={best_ckpt["val_acc"]:.4f})')

with open(HIST_FILE, 'w') as f:
    json.dump(history, f, indent=2)
print(f'History saved to {HIST_FILE}')

In [ ]:
# ============================================================
# CELL 10: LEARNING CURVES
# ============================================================
with open(HIST_FILE, 'r') as f:
    h = json.load(f)

epochs_list  = [e['epoch']      for e in h]
train_losses = [e['train_loss'] for e in h]
val_losses   = [e['val_loss']   for e in h]
val_accs     = [e['val_acc']    for e in h]
val_f1s      = [e['val_f1']     for e in h]
lrs          = [e['lr']         for e in h]

# Màu theo phase
warmup_eps = [e['epoch'] for e in h if e['phase'] == 'WARMUP']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Loss
axes[0,0].plot(epochs_list, train_losses, 'b-', label='Train Loss', linewidth=1.5)
axes[0,0].plot(epochs_list, val_losses,   'r-', label='Val Loss',   linewidth=1.5)
if warmup_eps:
    axes[0,0].axvspan(warmup_eps[0], warmup_eps[-1], alpha=0.1, color='orange', label='Warmup')
axes[0,0].set_title('Loss Curves'); axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

# Accuracy
axes[0,1].plot(epochs_list, val_accs, 'g-', linewidth=2)
axes[0,1].set_title('Validation Accuracy'); axes[0,1].set_ylim([0, 1])
axes[0,1].axhline(max(val_accs), color='g', linestyle='--', alpha=0.5, label=f'Best={max(val_accs):.4f}')
axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

# F1
axes[1,0].plot(epochs_list, val_f1s, 'm-', linewidth=2)
axes[1,0].set_title('Validation F1 (Macro)')
axes[1,0].axhline(max(val_f1s), color='m', linestyle='--', alpha=0.5, label=f'Best={max(val_f1s):.4f}')
axes[1,0].legend(); axes[1,0].set_ylim([0, 1]); axes[1,0].grid(True, alpha=0.3)

# LR schedule
axes[1,1].plot(epochs_list, lrs, 'c-', linewidth=2)
axes[1,1].set_title('Learning Rate Schedule'); axes[1,1].set_yscale('log')
axes[1,1].grid(True, alpha=0.3)

plt.suptitle('v4.0 Training Curves — DermaFusion 24-class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('v4_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved v4_learning_curves.png')

In [ ]:
# ============================================================
# CELL 11: EVALUATION với TTA
# ============================================================
# FIX v4.0: Test-Time Augmentation — average predictions từ nhiều transforms
# → cải thiện đáng kể độ chính xác trên validation set

print('Running TTA evaluation...')
model.eval()

# Lấy raw PIL images từ val_ds để áp dụng nhiều transform
# Dùng val dataset với từng tta transform, rồi average softmax
tta_logits_all = None  # shape (N, C)

for t_idx, tta_tfm in enumerate(tta_tfms_list):
    tta_ds = datasets.ImageFolder(val_dir, transform=tta_tfm)
    tta_loader = DataLoader(
        tta_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True
    )

    logits_list = []
    with torch.no_grad():
        for images, _ in tqdm(tta_loader, desc=f'TTA {t_idx+1}/{len(tta_tfms_list)}', leave=False):
            images = images.to(device, non_blocking=True)
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                logits = model(images)
            logits_list.append(logits.float().cpu())

    logits_tensor = torch.cat(logits_list, dim=0)  # (N, C)
    probs = torch.softmax(logits_tensor, dim=1)

    if tta_logits_all is None:
        tta_logits_all = probs
    else:
        tta_logits_all = tta_logits_all + probs

# Average và lấy prediction
tta_logits_all /= len(tta_tfms_list)
tta_preds  = tta_logits_all.argmax(1).numpy()
true_labels = np.array([y for _, y in tta_ds.samples])

tta_f1  = f1_score(true_labels, tta_preds, average='macro', zero_division=0)
tta_acc = (tta_preds == true_labels).mean()

print(f'\n=== TTA Evaluation Results ===')
print(f'TTA Accuracy : {tta_acc:.4f}')
print(f'TTA F1 Macro : {tta_f1:.4f}')
print(f'(Without TTA : Acc={best_ckpt["val_acc"]:.4f}, F1={best_ckpt["val_f1"]:.4f})')

In [ ]:
# ============================================================
# CELL 12: CONFUSION MATRIX & CLASSIFICATION REPORT
# ============================================================
cm = confusion_matrix(true_labels, tta_preds)

# FIX v4.0: Normalized confusion matrix — dễ đọc hơn khi class imbalanced
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
cm_norm = np.nan_to_num(cm_norm)

fig, axes = plt.subplots(1, 2, figsize=(36, 14))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title('Confusion Matrix (Raw Counts)', fontsize=12)
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
plt.setp(axes[0].get_xticklabels(), rotation=45, ha='right', fontsize=8)
plt.setp(axes[0].get_yticklabels(), rotation=0,  fontsize=8)

# Normalized
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            vmin=0, vmax=1, ax=axes[1])
axes[1].set_title('Confusion Matrix (Normalized per Row)', fontsize=12)
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')
plt.setp(axes[1].get_xticklabels(), rotation=45, ha='right', fontsize=8)
plt.setp(axes[1].get_yticklabels(), rotation=0,  fontsize=8)

plt.suptitle('v4.0 Confusion Matrix — DermaFusion 24-class (TTA)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('v4_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved v4_confusion_matrix.png')

# Classification Report
report = classification_report(true_labels, tta_preds,
                                target_names=class_names, zero_division=0)
print('\n' + report)

with open('v4_classification_report.txt', 'w') as f:
    f.write(f'v4.0 Classification Report (TTA)\n')
    f.write(f'Acc={tta_acc:.4f}, F1_macro={tta_f1:.4f}\n\n')
    f.write(report)
print('Saved v4_classification_report.txt')

In [ ]:
# ============================================================
# CELL 13: PER-CLASS F1 TREND (theo epoch)
# ============================================================
# Xem class nào cải thiện/xấu đi theo thời gian
# Hữu ích để detect overfitting trên class cụ thể

with open(HIST_FILE, 'r') as f:
    h = json.load(f)

# Chỉ plot 8 class khó nhất (F1 thấp nhất ở epoch cuối)
last_ep = h[-1]
pcf1 = last_ep.get('per_class_f1', {})
if pcf1:
    worst8 = sorted(pcf1.items(), key=lambda x: x[1])[:8]
    worst8_names = [cn for cn, _ in worst8]

    fig, ax = plt.subplots(figsize=(14, 6))
    for cn in worst8_names:
        f1_trend = [e['per_class_f1'].get(cn, 0) for e in h if 'per_class_f1' in e]
        ep_trend = [e['epoch'] for e in h if 'per_class_f1' in e]
        ax.plot(ep_trend, f1_trend, marker='o', markersize=3, label=cn, linewidth=1.5)

    ax.set_title('Per-Class F1 Trend (8 hardest classes)')
    ax.set_xlabel('Epoch'); ax.set_ylabel('F1 Score')
    ax.set_ylim([0, 1]); ax.legend(loc='lower right', fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('v4_perclass_f1_trend.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved v4_perclass_f1_trend.png')
else:
    print('per_class_f1 không có trong history (có thể là checkpoint cũ)')

In [ ]:
# ============================================================
# CELL 14: SUMMARY
# ============================================================
print('=' * 60)
print('v4.0 TRAINING SUMMARY')
print('=' * 60)
print(f'Dataset      : ISIC2019 + DermNet + PAD-UFES-20 (24 classes)')
print(f'Model        : EfficientNet-B4 (ImageNet pretrained)')
print(f'IMG_SIZE     : {IMG_SIZE}x{IMG_SIZE}')
print(f'Batch size   : {BATCH_SIZE} (effective: {BATCH_SIZE * ACCUM_STEPS})')
print(f'Epochs ran   : {len(history)}/{NUM_EPOCHS}')
print(f'Best epoch   : {best_ckpt["epoch"]}')
print(f'Val F1       : {best_ckpt["val_f1"]:.4f} (no TTA)')
print(f'Val Acc      : {best_ckpt["val_acc"]:.4f} (no TTA)')
print(f'TTA F1       : {tta_f1:.4f}')
print(f'TTA Acc      : {tta_acc:.4f}')
print(f'Checkpoint   : {OUT_NAME}')
print('='*60)
print('\nOutput files:')
for fname in [OUT_NAME, HIST_FILE,
              'v4_learning_curves.png',
              'v4_confusion_matrix.png',
              'v4_classification_report.txt',
              'v4_perclass_f1_trend.png']:
    exists = '✅' if os.path.exists(fname) else '❌'
    print(f'  {exists} {fname}')